# Transformer Kernel Demo (RTX 3050)

This notebook demonstrates **transformer building-block kernels** in one place:
- **Fused QKV**: y = x @ W + b (one matmul for Q, K, V projections)
- **Softmax**: row-wise softmax
- **LayerNorm**: normalize last dim, scale & shift
- **GELU**: activation
- **Fused MLP**: linear → GELU → linear

Compared: **PyTorch** (reference), **Triton**, and **CUDA** (when extension is built).

**Run from repo root.**

In [ ]:
import sys
import time
from pathlib import Path
root = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Algorithm: Transformer block kernels

| Kernel | Formula / role |
|--------|-----------------|
| **QKV** | Single matmul: (M, H) @ (H, 3H) + b → (M, 3H); then split into Q, K, V in attention. |
| **Softmax** | exp(x - max) / sum(exp) over last dim. |
| **LayerNorm** | (x - μ) * (1/√(σ²+ε)) * γ + β. |
| **GELU** | 0.5·x·(1+tanh(√(2/π)(x+0.044715·x³))). |
| **MLP** | h = GELU(x@W1+b1); out = h@W2+b2. |

All run in **FP16** with FP32 accumulation where needed.

## Example: Run all kernels (PyTorch reference)

In [ ]:
from benchmarks.transformer_reference import (
    fused_qkv_pytorch,
    softmax_pytorch,
    layernorm_pytorch,
    gelu_pytorch,
    fused_mlp_pytorch,
)

B, S, H = 2, 64, 128
M = B * S
H3, H4 = H * 3, H * 4
dtype = torch.float16

x = torch.randn(M, H, device="cuda", dtype=dtype)
w_qkv = torch.randn(H, H3, device="cuda", dtype=dtype)
b_qkv = torch.randn(H3, device="cuda", dtype=dtype)
y_qkv = fused_qkv_pytorch(x, w_qkv, b_qkv)
print("QKV out:", y_qkv.shape)

x_s = torch.randn(M, H, device="cuda", dtype=dtype)
y_sm = softmax_pytorch(x_s, dim=-1)
print("Softmax out:", y_sm.shape)

w_ln = torch.ones(H, device="cuda", dtype=dtype)
b_ln = torch.zeros(H, device="cuda", dtype=dtype)
y_ln = layernorm_pytorch(x_s, w_ln, b_ln)
print("LayerNorm out:", y_ln.shape)

w1 = torch.randn(H, H4, device="cuda", dtype=dtype)
b1 = torch.randn(H4, device="cuda", dtype=dtype)
w2 = torch.randn(H4, H, device="cuda", dtype=dtype)
b2 = torch.randn(H, device="cuda", dtype=dtype)
y_mlp = fused_mlp_pytorch(x, w1, b1, w2, b2)
print("MLP out:", y_mlp.shape)

## Benchmark: PyTorch vs Triton vs CUDA

In [ ]:
def run_bench(fn, warmup=5, repeat=15):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeat):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000 / repeat

B, S, H = 8, 256, 768
M = B * S
H3, H4 = H * 3, H * 4
dtype = torch.float16

results = []
x = torch.randn(M, H, device="cuda", dtype=dtype)
w_qkv = torch.randn(H, H3, device="cuda", dtype=dtype)
b_qkv = torch.randn(H3, device="cuda", dtype=dtype)
x_ln = torch.randn(M, H, device="cuda", dtype=dtype)
w_ln = torch.ones(H, device="cuda", dtype=dtype)
b_ln = torch.zeros(H, device="cuda", dtype=dtype)
w1 = torch.randn(H, H4, device="cuda", dtype=dtype)
b1 = torch.randn(H4, device="cuda", dtype=dtype)
w2 = torch.randn(H4, H, device="cuda", dtype=dtype)
b2 = torch.randn(H, device="cuda", dtype=dtype)

kernels = [
    ("QKV", lambda: fused_qkv_pytorch(x, w_qkv, b_qkv), "triton", "cuda", lambda: __import__("triton_kernels.qkv", fromlist=["fused_qkv_triton"]).fused_qkv_triton(x, w_qkv, b_qkv), lambda: __import__("gpu_kernels.transformer.transformer_cuda", fromlist=["fused_qkv_cuda"]).fused_qkv_cuda(x, w_qkv, b_qkv)),
    ("Softmax", lambda: softmax_pytorch(x_ln, -1), "triton", "cuda", lambda: __import__("triton_kernels.softmax", fromlist=["softmax_triton"]).softmax_triton(x_ln, -1), lambda: __import__("gpu_kernels.transformer.transformer_cuda", fromlist=["softmax_cuda"]).softmax_cuda(x_ln, -1)),
    ("LayerNorm", lambda: layernorm_pytorch(x_ln, w_ln, b_ln), "triton", "cuda", lambda: __import__("triton_kernels.layernorm", fromlist=["layernorm_triton"]).layernorm_triton(x_ln, (H,), w_ln, b_ln), lambda: __import__("gpu_kernels.transformer.transformer_cuda", fromlist=["layernorm_cuda"]).layernorm_cuda(x_ln, w_ln, b_ln, 1e-5)),
    ("GELU", lambda: gelu_pytorch(x_ln.reshape(-1)), "triton", "cuda", lambda: __import__("triton_kernels.gelu", fromlist=["gelu_triton"]).gelu_triton(x_ln.reshape(-1)), lambda: __import__("gpu_kernels.transformer.transformer_cuda", fromlist=["gelu_cuda"]).gelu_cuda(x_ln.reshape(-1))),
    ("MLP", lambda: fused_mlp_pytorch(x, w1, b1, w2, b2), "triton", "cuda", lambda: __import__("triton_kernels.mlp", fromlist=["fused_mlp_triton"]).fused_mlp_triton(x, w1, b1, w2, b2), lambda: __import__("gpu_kernels.transformer.transformer_cuda", fromlist=["fused_mlp_cuda"]).fused_mlp_cuda(x, w1, b1, w2, b2)),
]

for name, fn_pt, _, _, fn_tr, fn_cuda in kernels:
    t_pt = run_bench(fn_pt)
    results.append((name, "PyTorch", t_pt))
    try:
        t_tr = run_bench(fn_tr)
        results.append((name, "Triton", t_tr))
    except Exception:
        pass
    try:
        out = fn_cuda()
        if out is not None:
            t_cuda = run_bench(fn_cuda)
            results.append((name, "CUDA", t_cuda))
    except Exception:
        pass

for name, impl, ms in results:
    print(f"  {name} {impl}: {ms:.4f} ms")

## Visualize performance

In [ ]:
import matplotlib.pyplot as plt

kernels = sorted(set(r[0] for r in results))
impls = ["PyTorch", "Triton", "CUDA"]
data = {impl: [] for impl in impls}
for k in kernels:
    for impl in impls:
        t = next((r[2] for r in results if r[0] == k and r[1] == impl), None)
        data[impl].append(t if t is not None else 0)

x = range(len(kernels))
w = 0.25
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - w for i in x], data["PyTorch"], w, label="PyTorch", color="#2ecc71")
ax.bar([i for i in x], data["Triton"], w, label="Triton", color="#3498db")
ax.bar([i + w for i in x], data["CUDA"], w, label="CUDA", color="#9b59b6")
ax.set_xticks(x)
ax.set_xticklabels(kernels)
ax.set_ylabel("Latency (ms)")
ax.set_title("Transformer kernels: PyTorch vs Triton vs CUDA (B=8, S=256, H=768)")
ax.legend()
plt.tight_layout()
plt.show()